# Accident classifier training (frozen MobileNetV3-Small + trainable LSTM)
Before running, put these in Google Drive:
- `road_incident/` containing the project's `src/` and `training/` folders and your `manifest.csv`
- the TADS videos (see README for how to obtain them)

Runtime > Change runtime type > GPU. Then run the cells top to bottom.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Edit these paths to match your Drive layout
PROJECT_DIR = "/content/drive/MyDrive/road_incident"   # contains src/ and training/
DATA_DIR    = "/content/drive/MyDrive/TADS"            # folder holding the dataset videos
MANIFEST    = f"{PROJECT_DIR}/manifest.csv"            # columns: video,label,start_sec,end_sec
CLIPS_DIR   = "/content/clips"                         # local disk (faster than Drive)
OUT_CKPT    = f"{PROJECT_DIR}/accident_cnn_lstm.pt"
EPOCHS      = 8
%cd {PROJECT_DIR}

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())

## 1. Prepare clips (frame sequences, no train/val/test mixing yet)

In [ ]:
!python training/prepare_clips.py --manifest "{MANIFEST}" --video-root "{DATA_DIR}" --out {CLIPS_DIR}

## 2. Train, split by source video, evaluate on held-out test videos

In [ ]:
!python training/train.py --clips {CLIPS_DIR}/clips.csv --out "{OUT_CKPT}" --epochs {EPOCHS}

## 3. Metrics and checkpoint download

In [ ]:
import json
ck = torch.load(OUT_CKPT, map_location="cpu")
print(json.dumps(ck["metrics"], indent=2))
from google.colab import files
files.download(OUT_CKPT)